# Habituation / repeated-exposure analysis

This notebook generates the final analysis tables and draft figures for the habituation phase of the AIR Wheel behavioral experiment.

The goal is to quantify whether repeated head-fixed wheel exposure changes:

1. **whole-session locomotor state** across habituation sessions,
2. **within-session locomotor dynamics** across time in each session, and
3. whether the results are robust to animal-level variability.

Primary outcomes are restricted to the three measures reported in the manuscript:

- fraction moving,
- fraction forward,
- speed, defined as path speed in cm/s.

## 1. Analysis settings

These constants define the primary outcomes, labels, habituation phase name, movement threshold, and the common within-session analysis window.

In [ ]:
PRIMARY_OUTCOMES = [
    "frac_moving",
    "frac_forward",
    "mean_speed_path_cms",
]

OUTCOME_LABELS = {
    "frac_moving": "Fraction moving",
    "frac_forward": "Fraction forward",
    "mean_speed_path_cms": "Speed (cm/s)",
}

HAB_PHASE = "habituation"
SPEED_THRESH = 0.5
MAX_SESSION_TIME_MIN = 20.0

## 2. Load project context and cached behavioral epoch data

This section loads the project paths, epoch-window tables, behavioral epoch metrics, QC tables, and session-day metadata. The session-day table is used to identify good sessions and assign phase/session numbers.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Find the project root that contains the src folder
current = Path.cwd().resolve()

for p in [current] + list(current.parents):
    if (p / "src").exists():
        repo_root = p
        break
else:
    raise FileNotFoundError("Could not find a folder containing 'src'.")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("Current working directory:", current)
print("Added repo root:", repo_root)
print("src exists:", (repo_root / "src").exists())

from pathlib import Path
import numpy as np
import pandas as pd

import src.utils.pdata_io as pdio
from src.proc.extract_epoch_windows import load_epoch_windows

data_root, pdata_root, cc_data = pdio.load_project_context()

windows_df = load_epoch_windows(
    pdata_root=pdata_root,
    filename="behavior_epoch_windows.h5",
    key="windows/prepost_1s"
)

valid_windows = windows_df[windows_df["valid_window"]].copy()

print("All windows:", windows_df.shape)
print("Valid windows:", valid_windows.shape)

valid_windows.groupby(["phase", "epoch_name"]).size().reset_index(name="n_windows")

encoder_epoch_df = pd.read_hdf(
    Path(pdata_root) / "_cache" / "behavior_epoch_metrics.h5",
    key="encoder/prepost_1s_speedThresh_1cms"
)

from src.qc.qc_events import load_behavior_qc_tables

events_df, session_summary_df = load_behavior_qc_tables(
    pdata_root=pdata_root,
    filename="behavior_QC.h5"
)

from src.utils.pdata_organize import make_session_availability_summary

session_availability_df = make_session_availability_summary(
    events_df=events_df,
    windows_df=windows_df,
    min_session_duration_s=900,
    min_valid_events=3,
)

from src.utils.pdata_organize import add_day_bins_to_sessions

session_day_df = add_day_bins_to_sessions(
    session_availability_df,
    use_good_sessions_only=True,
)

### Inspect session-day metadata

This quick display is mainly a sanity check to confirm that session availability and phase/session numbering were loaded correctly.

In [ ]:
session_day_df

## 3. Define output folder for figures and tables

All draft figures and model-output tables are saved into the processed-data figure directory.

In [ ]:
from pathlib import Path

# Make figures folder inside pdata directory
fig_dir = Path(pdata_root) / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

# Convert to string for R compatibility
fig_dir_str = str(fig_dir)
fig_dir

## 4. Whole-session habituation dataframe

Here we compute one row per animal-session using whole-session locomotor metrics. This dataframe is used for the first Results section: repeated exposure effects across habituation sessions.

In [ ]:
from src.proc.behavior_metrics import compute_whole_session_locomotor_metrics

hab_df = compute_whole_session_locomotor_metrics(
    session_day_df=session_summary_df,
    phase="habituation",
    speed_thresh=0.5,
    good_sessions_only=True,
)

hab_df.head()

### Save and reload the whole-session dataframe

The whole-session metrics are saved to HDF5 so that the analysis can be reproduced without recomputing them every time.

In [ ]:
from pathlib import Path
import src.utils.pdata_io as pdio

data_root, pdata_root, cc_data = pdio.load_project_context()

out_file = Path(pdata_root) / "_cache" / "behavior_whole_session_metrics.h5"

pdio.save_df_h5(
    df=hab_df,
    h5_path=out_file,
    key="whole_session/habituation_speedThresh_0p5",
    overwrite=True,
    metadata={
        "phase": "habituation",
        "speed_thresh": 0.5,
        "good_sessions_only": True,
        "description": "Whole-session locomotor metrics for habituation phase",
    },
)

In [ ]:
out_file = Path(pdata_root) / "_cache" / "behavior_whole_session_metrics.h5"

hab_df_loaded = pd.read_hdf(
    out_file,
    key="whole_session/habituation_speedThresh_0p5"
)

hab_df_loaded.head()

hab_df = hab_df_loaded.copy()

## 5. Prepare whole-session dataframe for LME

This section adds a unique animal-day/session identifier and creates `exposure_session_c`.

`exposure_session_c` is the exposure session number centered at the mean session number. The slope is still interpreted as change per additional exposure session.

In [ ]:
hab_session_df = hab_df.copy()

# Animal-day/session ID
hab_session_df["animal_day"] = (
    hab_session_df["animal"].astype(str) + ":" +
    hab_session_df["date"].astype(str)
)

# Exposure session number
if "exposure_session" not in hab_session_df.columns:
    if "phase_session_number" in hab_session_df.columns:
        hab_session_df["exposure_session"] = hab_session_df["phase_session_number"]
    elif "phase_day_number_good" in hab_session_df.columns:
        hab_session_df["exposure_session"] = hab_session_df["phase_day_number_good"]
    elif "phase_day" in hab_session_df.columns:
        hab_session_df["exposure_session"] = hab_session_df["phase_day"]
    else:
        raise ValueError("Could not find exposure/session/day column.")

hab_session_df["exposure_session"] = hab_session_df["exposure_session"].astype(float)

hab_session_df["exposure_session_c"] = (
    hab_session_df["exposure_session"] -
    hab_session_df["exposure_session"].mean()
)

print(hab_session_df.shape)
print(hab_session_df.columns.tolist())
display(hab_session_df.head())

## 6. Enable R in the notebook

The mixed-effects models are fit in R using `lme4`/`lmerTest`, while Python is used for data organization and saving tables.

In [ ]:
%load_ext rpy2.ipython

## 7. Whole-session LME models

For each primary outcome, we fit:

`outcome ~ exposure_session_c + (1 | animal)`

This model asks whether the outcome changes across habituation sessions while allowing each animal to have its own baseline level.

In [ ]:
%%R -i hab_session_df -o hab_session_results_R

library(lme4)
library(lmerTest)
library(broom.mixed)
library(dplyr)

# --------------------------------------------------
# Prepare factors
# --------------------------------------------------

hab_session_df$animal <- factor(hab_session_df$animal)

# --------------------------------------------------
# Primary outcomes
# --------------------------------------------------

primary_outcomes <- c(
  "frac_moving",
  "frac_forward",
  "mean_speed_path_cms"
)

primary_outcomes <- primary_outcomes[primary_outcomes %in% names(hab_session_df)]

print(primary_outcomes)

# --------------------------------------------------
# Whole-session models
# outcome ~ exposure_session_c + (1 | animal)
# --------------------------------------------------

session_results <- list()
session_models <- list()

for (outcome in primary_outcomes) {

  cat("\n\n==============================\n")
  cat("Fitting:", outcome, "\n")
  cat("==============================\n")

  formula_text <- paste0(
    outcome,
    " ~ exposure_session_c + (1 | animal)"
  )

  m <- lmer(
    as.formula(formula_text),
    data = hab_session_df,
    REML = FALSE
  )

  session_models[[outcome]] <- m

  tmp <- broom.mixed::tidy(
    m,
    effects = "fixed",
    conf.int = TRUE
  )

  tmp$outcome <- outcome
  session_results[[outcome]] <- tmp

  print(summary(m))
}

hab_session_results_R <- bind_rows(session_results) %>%
  relocate(outcome)

print(hab_session_results_R)

### Format whole-session model results for manuscript reporting

This converts the R fixed-effect output into compact manuscript-style statistical text.

In [ ]:
import pandas as pd
import numpy as np

def format_p(p):
    if pd.isna(p):
        return "p = NA"
    elif p < 0.001:
        return "p < 0.001"
    else:
        return f"p = {p:.3f}"

def make_lme_report_table(df):
    report_df = df.copy()

    report_df["report_text"] = report_df.apply(
        lambda r: (
            f"β = {r['estimate']:.4f}, "
            f"SE = {r['std.error']:.4f}, "
            f"t({r['df']:.1f}) = {r['statistic']:.2f}, "
            f"{format_p(r['p.value'])}"
        ),
        axis=1
    )

    return report_df[
        ["outcome", "term", "estimate", "std.error", "statistic", "df", "p.value", "report_text"]
    ]

hab_session_report = make_lme_report_table(hab_session_results_R)

display(
    hab_session_report[
        hab_session_report["term"] != "(Intercept)"
    ]
)

## 8. Figure 1A–C: whole-session repeated-exposure effects

This figure shows the raw animal-session values, individual animal trajectories, and the fixed-effect LME prediction for each primary outcome.

The figure is saved as PDF/PNG in the figure directory.

In [ ]:
%%R -i hab_session_df -i fig_dir_str

library(lme4)
library(lmerTest)
library(dplyr)
library(ggplot2)
library(patchwork)

# --------------------------------------------------
# Output folder
# --------------------------------------------------

out_dir <- fig_dir_str
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

# --------------------------------------------------
# Prepare factors
# --------------------------------------------------

hab_session_df$animal <- factor(hab_session_df$animal)

# --------------------------------------------------
# Primary outcomes
# --------------------------------------------------

primary_outcomes <- c(
  "frac_moving",
  "frac_forward",
  "mean_speed_path_cms"
)

primary_outcomes <- primary_outcomes[primary_outcomes %in% names(hab_session_df)]

outcome_labels <- c(
  frac_moving = "Fraction moving",
  frac_forward = "Fraction forward",
  mean_speed_path_cms = "Speed (cm/s)"
)

# --------------------------------------------------
# Helper: p-value formatting
# --------------------------------------------------

format_p <- function(p) {
  if (is.na(p)) {
    return("p = NA")
  } else if (p < 0.001) {
    return("p < 0.001")
  } else {
    return(paste0("p = ", sprintf("%.3f", p)))
  }
}

# --------------------------------------------------
# Helper: fixed-effect prediction with 95% CI
# --------------------------------------------------

predict_fixed_ci <- function(model, newdata) {
  fixed_formula <- formula(model, fixed.only = TRUE)
  X <- model.matrix(delete.response(terms(fixed_formula)), newdata)
  beta <- fixef(model)
  V <- vcov(model)

  fit <- as.numeric(X %*% beta)
  se <- sqrt(diag(X %*% V %*% t(X)))

  newdata$fit <- fit
  newdata$se <- se
  newdata$lower <- fit - 1.96 * se
  newdata$upper <- fit + 1.96 * se

  return(newdata)
}

# --------------------------------------------------
# Theme
# --------------------------------------------------

theme_fig <- function() {
  theme_classic(base_size = 7) +
    theme(
      axis.title = element_text(size = 7),
      axis.text = element_text(size = 6),
      plot.title = element_blank(),
      plot.subtitle = element_blank(),
      legend.position = "none",
      plot.margin = margin(3, 5, 3, 3)
    )
}

# --------------------------------------------------
# Fit models and make one plot per outcome
# --------------------------------------------------

plot_list <- list()
stats_list <- list()

for (outcome in primary_outcomes) {

  label_i <- unname(outcome_labels[outcome])

  m <- lmer(
    as.formula(paste0(outcome, " ~ exposure_session_c + (1 | animal)")),
    data = hab_session_df,
    REML = FALSE
  )

  sm <- summary(m)
  coef_table <- as.data.frame(sm$coefficients)

  beta <- coef_table["exposure_session_c", "Estimate"]
  se <- coef_table["exposure_session_c", "Std. Error"]
  df <- coef_table["exposure_session_c", "df"]
  tval <- coef_table["exposure_session_c", "t value"]
  pval <- coef_table["exposure_session_c", "Pr(>|t|)"]

  # Use plain "beta" to avoid Greek font/rendering problems
  stat_label <- paste0(
    "beta = ", sprintf("%.3f", beta), "/session\n",
    format_p(pval)
  )

  stats_list[[outcome]] <- data.frame(
    outcome = outcome,
    y_label = label_i,
    beta = beta,
    se = se,
    df = df,
    t = tval,
    p = pval,
    stat_label = stat_label
  )

  exposure_seq <- seq(
    min(hab_session_df$exposure_session, na.rm = TRUE),
    max(hab_session_df$exposure_session, na.rm = TRUE),
    length.out = 100
  )

  exposure_mean <- mean(hab_session_df$exposure_session, na.rm = TRUE)

  pred_df <- data.frame(
    exposure_session = exposure_seq,
    exposure_session_c = exposure_seq - exposure_mean
  )

  pred_df <- predict_fixed_ci(m, pred_df)

  raw_df <- hab_session_df %>%
    select(animal, exposure_session, all_of(outcome)) %>%
    rename(y = all_of(outcome))

  x_annot <- min(raw_df$exposure_session, na.rm = TRUE) +
    0.04 * diff(range(raw_df$exposure_session, na.rm = TRUE))

  y_annot <- max(raw_df$y, na.rm = TRUE) -
    0.06 * diff(range(raw_df$y, na.rm = TRUE))

  p <- ggplot() +
    geom_line(
      data = raw_df,
      aes(x = exposure_session, y = y, group = animal),
      alpha = 0.25,
      linewidth = 0.25
    ) +
    geom_point(
      data = raw_df,
      aes(x = exposure_session, y = y),
      alpha = 0.45,
      size = 0.8
    ) +
    geom_ribbon(
      data = pred_df,
      aes(x = exposure_session, ymin = lower, ymax = upper),
      alpha = 0.20
    ) +
    geom_line(
      data = pred_df,
      aes(x = exposure_session, y = fit),
      linewidth = 0.7
    ) +
    annotate(
      "text",
      x = x_annot,
      y = y_annot,
      label = stat_label,
      hjust = 0,
      vjust = 1,
      size = 2.1
    ) +
    labs(
      x = "Exposure session",
      y = label_i
    ) +
    theme_fig()

  plot_list[[outcome]] <- p
}

# --------------------------------------------------
# Combine panels
# --------------------------------------------------

p_combined <- plot_list[[primary_outcomes[1]]] +
  plot_list[[primary_outcomes[2]]] +
  plot_list[[primary_outcomes[3]]] +
  plot_layout(nrow = 1)

print(p_combined)

# --------------------------------------------------
# Save
# --------------------------------------------------

ggsave(
  filename = file.path(out_dir, "Fig1_whole_session_habituation_3panel.pdf"),
  plot = p_combined,
  width = 7,
  height = 1.9
)

ggsave(
  filename = file.path(out_dir, "Fig1_whole_session_habituation_3panel.png"),
  plot = p_combined,
  width = 7,
  height = 1.9,
  dpi = 600
)

# SVG should work after installing r-svglite
ggsave(
  filename = file.path(out_dir, "Fig1_whole_session_habituation_3panel.svg"),
  plot = p_combined,
  width = 7,
  height = 1.9
)

# --------------------------------------------------
# Save stats table
# --------------------------------------------------

hab_whole_session_stats_df <- bind_rows(stats_list)

write.csv(
  hab_whole_session_stats_df,
  file.path(out_dir, "habituation_whole_session_LME_stats.csv"),
  row.names = FALSE
)

print(hab_whole_session_stats_df)

## 9. Animal-level sensitivity analysis

The whole-session figure shows that one animal has a particularly large increase, especially for speed. To check whether the repeated-exposure effect is driven only by this high-responder, we estimate a separate slope for each animal and summarize how many animals show a positive slope.

In [ ]:
%%R -i hab_session_df -o animal_slope_results_df -o animal_slope_summary_df

library(dplyr)

primary_outcomes <- c(
  "frac_moving",
  "frac_forward",
  "mean_speed_path_cms"
)

primary_outcomes <- primary_outcomes[primary_outcomes %in% names(hab_session_df)]

animal_slope_results <- list()

for (outcome in primary_outcomes) {

  tmp <- hab_session_df %>%
    group_by(animal) %>%
    group_modify(~{
      fit <- lm(as.formula(paste0(outcome, " ~ exposure_session")), data = .x)
      coefs <- summary(fit)$coefficients

      data.frame(
        n_sessions = nrow(.x),
        df = df.residual(fit),
        beta = coefs["exposure_session", "Estimate"],
        se = coefs["exposure_session", "Std. Error"],
        t = coefs["exposure_session", "t value"],
        p = coefs["exposure_session", "Pr(>|t|)"]
      )
    }) %>%
    ungroup() %>%
    mutate(outcome = outcome)

  animal_slope_results[[outcome]] <- tmp
}

animal_slope_results_df <- bind_rows(animal_slope_results)

animal_slope_summary_df <- animal_slope_results_df %>%
  group_by(outcome) %>%
  summarise(
    n_animals = n(),
    n_positive = sum(beta > 0),
    mean_beta = mean(beta, na.rm = TRUE),
    median_beta = median(beta, na.rm = TRUE),
    min_beta = min(beta, na.rm = TRUE),
    max_beta = max(beta, na.rm = TRUE),
    .groups = "drop"
  )

print(animal_slope_results_df)
print(animal_slope_summary_df)

### Leave-one-animal-out sensitivity analysis

This optional sensitivity analysis refits the whole-session LME after removing one animal at a time. The goal is to determine whether the exposure-session effect remains positive when any single animal is excluded.

In [ ]:
%%R -i hab_session_df

library(lme4)
library(lmerTest)
library(dplyr)

hab_session_df$animal <- factor(hab_session_df$animal)

primary_outcomes <- c(
  "frac_moving",
  "frac_forward",
  "mean_speed_path_cms"
)

primary_outcomes <- primary_outcomes[primary_outcomes %in% names(hab_session_df)]

loo_results <- list()

for (outcome in primary_outcomes) {

  animals <- levels(hab_session_df$animal)

  for (a in animals) {

    df_sub <- hab_session_df %>%
      filter(animal != a)

    m <- lmer(
      as.formula(paste0(outcome, " ~ exposure_session_c + (1 | animal)")),
      data = df_sub,
      REML = FALSE
    )

    coef_table <- as.data.frame(summary(m)$coefficients)

    loo_results[[paste(outcome, a, sep = "_")]] <- data.frame(
      outcome = outcome,
      animal_removed = a,
      beta = coef_table["exposure_session_c", "Estimate"],
      se = coef_table["exposure_session_c", "Std. Error"],
      df = coef_table["exposure_session_c", "df"],
      t = coef_table["exposure_session_c", "t value"],
      p = coef_table["exposure_session_c", "Pr(>|t|)"]
    )
  }
}

loo_results_df <- bind_rows(loo_results)

print(loo_results_df)

## 10. Build epoch-level habituation dataframe

The within-session analysis requires an epoch-level dataframe, not the whole-session dataframe. Here we extract habituation epochs from `encoder_epoch_df`, add animal-day IDs, add exposure-session numbers, add within-session time, and keep only the primary outcomes.

`session_time_10m_c` is within-session time centered at the mean sampled time and scaled by 10 minutes. Its coefficient is interpreted as change per 10 minutes later within a session.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# --------------------------------------------------
# Build habituation epoch-level dataframe
# --------------------------------------------------

df0 = encoder_epoch_df.copy()

print("Available phases:")
print(df0["phase"].value_counts())

# Detect habituation phase
phase_vals = df0["phase"].dropna().astype(str).unique().tolist()

if "habituation" in phase_vals:
    hab_phase = "habituation"
else:
    hab_candidates = [
        p for p in phase_vals
        if ("hab" in p.lower()) or ("exposure" in p.lower())
    ]
    if len(hab_candidates) == 0:
        raise ValueError(f"Could not identify habituation phase. Available phases: {phase_vals}")
    hab_phase = hab_candidates[0]

print("Using habituation phase:", hab_phase)

hab_epoch_df = df0[df0["phase"].astype(str) == hab_phase].copy()

# --------------------------------------------------
# Apply validity filters if columns exist
# --------------------------------------------------

if "valid_window" in hab_epoch_df.columns:
    hab_epoch_df = hab_epoch_df[hab_epoch_df["valid_window"] == True].copy()

if "good_session_basic" in hab_epoch_df.columns:
    hab_epoch_df = hab_epoch_df[hab_epoch_df["good_session_basic"] == True].copy()

# --------------------------------------------------
# Animal-day/session ID
# --------------------------------------------------

if "date" in hab_epoch_df.columns:
    hab_epoch_df["animal_day"] = (
        hab_epoch_df["animal"].astype(str) + ":" +
        hab_epoch_df["date"].astype(str)
    )
else:
    raise ValueError("date column not found; need a session/date identifier.")

# --------------------------------------------------
# Add exposure session number
# --------------------------------------------------

if "exposure_session" not in hab_epoch_df.columns:
    if "phase_session_number" in hab_epoch_df.columns:
        hab_epoch_df["exposure_session"] = hab_epoch_df["phase_session_number"]
    elif "phase_day_number_good" in hab_epoch_df.columns:
        hab_epoch_df["exposure_session"] = hab_epoch_df["phase_day_number_good"]
    elif "phase_day" in hab_epoch_df.columns:
        hab_epoch_df["exposure_session"] = hab_epoch_df["phase_day"]
    else:
        # Try merging from session_day_df if available
        if "session_day_df" in globals():
            merge_cols = ["animal", "date"]
            possible_session_cols = [
                "phase_session_number",
                "phase_day_number_good",
                "phase_day",
                "training_day",
            ]
            available_session_cols = [
                c for c in possible_session_cols if c in session_day_df.columns
            ]

            if len(available_session_cols) == 0:
                raise ValueError(
                    "No session/day column found in hab_epoch_df or session_day_df."
                )

            session_col = available_session_cols[0]

            hab_epoch_df = hab_epoch_df.merge(
                session_day_df[merge_cols + [session_col]].drop_duplicates(),
                on=merge_cols,
                how="left"
            )

            hab_epoch_df["exposure_session"] = hab_epoch_df[session_col]
        else:
            raise ValueError(
                "Could not find exposure/session/day column and session_day_df is not available."
            )

hab_epoch_df["exposure_session"] = hab_epoch_df["exposure_session"].astype(float)

# --------------------------------------------------
# Add within-session time in minutes
# --------------------------------------------------

if "session_time_min" not in hab_epoch_df.columns:
    if "anchor_session_time_min" in hab_epoch_df.columns:
        hab_epoch_df["session_time_min"] = hab_epoch_df["anchor_session_time_min"]
    elif "window_center_time_s" in hab_epoch_df.columns:
        hab_epoch_df["session_time_min"] = hab_epoch_df["window_center_time_s"] / 60.0
    elif "anchor_time_s" in hab_epoch_df.columns:
        hab_epoch_df["session_time_min"] = hab_epoch_df["anchor_time_s"] / 60.0
    elif "session_time_s" in hab_epoch_df.columns:
        hab_epoch_df["session_time_min"] = hab_epoch_df["session_time_s"] / 60.0
    else:
        print("Time-related columns:")
        print([c for c in hab_epoch_df.columns if "time" in c.lower()])
        raise ValueError("Could not find a within-session time column.")

# --------------------------------------------------
# Make centered predictors
# --------------------------------------------------

hab_epoch_df["exposure_session_c"] = (
    hab_epoch_df["exposure_session"] -
    hab_epoch_df["exposure_session"].mean()
)

hab_epoch_df["session_time_10m_c"] = (
    hab_epoch_df["session_time_min"] -
    hab_epoch_df["session_time_min"].mean()
) / 10.0

# --------------------------------------------------
# Make sure primary outcomes exist
# --------------------------------------------------

if "frac_moving" not in hab_epoch_df.columns:
    if "frac_stationary" in hab_epoch_df.columns:
        hab_epoch_df["frac_moving"] = 1 - hab_epoch_df["frac_stationary"]
    elif "stationary_frac" in hab_epoch_df.columns:
        hab_epoch_df["frac_moving"] = 1 - hab_epoch_df["stationary_frac"]

primary_outcomes = [
    "frac_moving",
    "frac_forward",
    "mean_speed_path_cms",
]

missing = [c for c in primary_outcomes if c not in hab_epoch_df.columns]
if len(missing) > 0:
    print("Missing primary outcomes:", missing)
    print("Available fraction columns:")
    print([c for c in hab_epoch_df.columns if "frac" in c.lower()])
    print("Available speed columns:")
    print([c for c in hab_epoch_df.columns if "speed" in c.lower()])
    raise ValueError("Some primary outcome columns are missing.")

# --------------------------------------------------
# Drop rows missing required values
# --------------------------------------------------

required_cols = [
    "animal",
    "animal_day",
    "exposure_session",
    "exposure_session_c",
    "session_time_min",
    "session_time_10m_c",
] + primary_outcomes

hab_epoch_df = hab_epoch_df.dropna(subset=required_cols).copy()

# --------------------------------------------------
# Summary
# --------------------------------------------------

print("hab_epoch_df shape:", hab_epoch_df.shape)
print("n animals:", hab_epoch_df["animal"].nunique())
print("n animal-day sessions:", hab_epoch_df["animal_day"].nunique())
print("n sampled epoch windows:", len(hab_epoch_df))

print("\nExposure session summary:")
display(hab_epoch_df["exposure_session"].describe())

print("\nSession time summary:")
display(hab_epoch_df["session_time_min"].describe())

print("\nMean session time subtracted:")
print(hab_epoch_df["session_time_min"].mean())

print("\nOutcome summaries:")
display(hab_epoch_df[primary_outcomes].describe())

display(hab_epoch_df.head())
hab_epoch_df_original = hab_epoch_df.copy()

## 11. Diagnose session duration

Before fitting within-session models, we inspect the time columns and the maximum session time per animal-day. This is important because one recording extends beyond the typical 20-minute session duration.

In [ ]:
# Check time-related columns in encoder_epoch_df and hab_epoch_df if it exists
for df_name in ["encoder_epoch_df", "hab_epoch_df", "windows_df", "valid_windows"]:
    if df_name in globals():
        df = globals()[df_name]
        print("\n", df_name, df.shape)
        print([c for c in df.columns if "time" in c.lower() or "anchor" in c.lower() or "window" in c.lower()])

In [ ]:
# Check per-session max within-session time
tmp = hab_epoch_df.copy()

session_time_check = (
    tmp.groupby(["animal", "date", "animal_day"], observed=True)
    .agg(
        n_windows=("session_time_min", "size"),
        min_time_min=("session_time_min", "min"),
        max_time_min=("session_time_min", "max"),
        mean_time_min=("session_time_min", "mean"),
    )
    .reset_index()
    .sort_values("max_time_min", ascending=False)
)

display(session_time_check.head(20))

print("Number of sessions with max time > 25 min:")
print((session_time_check["max_time_min"] > 25).sum())

display(session_time_check[session_time_check["max_time_min"] > 25])

## 12. Restrict within-session analysis to the common first 20 minutes

Most habituation sessions were approximately 20 minutes. One session extended to about 42 minutes, so the final within-session analysis is restricted to the first 20 minutes of each session to keep temporal coverage comparable across animals and sessions.

After filtering, `exposure_session_c` and `session_time_10m_c` are recalculated.

In [ ]:
# --------------------------------------------------
# Restrict within-session analysis to common 20-min window
# --------------------------------------------------

ANALYSIS_MAX_MIN = 20.0

hab_epoch_df_20min = hab_epoch_df[
    (hab_epoch_df["session_time_min"] >= 0) &
    (hab_epoch_df["session_time_min"] <= ANALYSIS_MAX_MIN)
].copy()

# Re-center after filtering
hab_epoch_df_20min["exposure_session_c"] = (
    hab_epoch_df_20min["exposure_session"] -
    hab_epoch_df_20min["exposure_session"].mean()
)

hab_epoch_df_20min["session_time_10m_c"] = (
    hab_epoch_df_20min["session_time_min"] -
    hab_epoch_df_20min["session_time_min"].mean()
) / 10.0

print("Original hab_epoch_df:", hab_epoch_df.shape)
print("Restricted hab_epoch_df_20min:", hab_epoch_df_20min.shape)

print("\nSession time summary after restriction:")
display(hab_epoch_df_20min["session_time_min"].describe())

print("\nMean session time subtracted:")
print(hab_epoch_df_20min["session_time_min"].mean())

print("\nNumber of animals:", hab_epoch_df_20min["animal"].nunique())
print("Number of sessions:", hab_epoch_df_20min["animal_day"].nunique())
print("Number of sampled windows:", len(hab_epoch_df_20min))

hab_epoch_df = hab_epoch_df_20min.copy()

## 13. Sensitivity check: unrestricted vs. first-20-min within-session models

This section fits the same epoch-level LME to both the unrestricted dataframe and the first-20-minute restricted dataframe. We compare the key terms:

- exposure session,
- within-session time,
- exposure session × within-session time.

This tells us whether the 20-minute restriction changes the reported model conclusions.

In [ ]:
%%R -i hab_epoch_df -i hab_epoch_df_20min -o hab_epoch_restriction_comparison_df

library(lme4)
library(lmerTest)
library(dplyr)

primary_outcomes <- c(
  "frac_moving",
  "frac_forward",
  "mean_speed_path_cms"
)

run_epoch_model_summary <- function(df, dataset_label) {

  df$animal <- factor(df$animal)
  df$animal_day <- factor(df$animal_day)

  results <- list()

  for (outcome in primary_outcomes) {

    if (!(outcome %in% names(df))) {
      next
    }

    formula_text <- paste0(
      outcome,
      " ~ exposure_session_c * session_time_10m_c + ",
      "(1 | animal) + (1 | animal_day)"
    )

    m <- lmer(
      as.formula(formula_text),
      data = df,
      REML = FALSE,
      control = lmerControl(
        optimizer = "bobyqa",
        optCtrl = list(maxfun = 2e5)
      )
    )

    coef_table <- as.data.frame(summary(m)$coefficients)

    tmp <- data.frame(
      dataset = dataset_label,
      outcome = outcome,
      term = rownames(coef_table),
      beta = coef_table[, "Estimate"],
      se = coef_table[, "Std. Error"],
      df = coef_table[, "df"],
      t = coef_table[, "t value"],
      p = coef_table[, "Pr(>|t|)"],
      n_obs = nobs(m),
      row.names = NULL
    )

    results[[outcome]] <- tmp
  }

  bind_rows(results)
}

unrestricted_df <- run_epoch_model_summary(hab_epoch_df, "unrestricted")
restricted_df <- run_epoch_model_summary(hab_epoch_df_20min, "first_20_min")

hab_epoch_restriction_comparison_df <- bind_rows(
  unrestricted_df,
  restricted_df
) %>%
  filter(term %in% c(
    "exposure_session_c",
    "session_time_10m_c",
    "exposure_session_c:session_time_10m_c"
  ))

print(hab_epoch_restriction_comparison_df)

## 14. Final within-session LME models and Figure 1D–F

For the final within-session analysis, we use the first-20-minute restricted dataframe and fit:

`outcome ~ exposure_session_c * session_time_10m_c + (1 | animal) + (1 | animal_day)`

This model asks whether locomotor state and speed change:

1. across exposure sessions,
2. across time within a session, and
3. whether the within-session time profile changes across exposure sessions.

The figure displays binned raw means and fixed-effect model predictions for early, mean, and late exposure. Interaction statistics are saved in the output table and reported in the Results/caption rather than written inside the plot panels.

In [ ]:
%%R -i hab_epoch_df_20min -i fig_dir_str -o hab_epoch_stats_df -o hab_speed_simple_slopes_t_df

library(lme4)
library(lmerTest)
library(dplyr)
library(ggplot2)
library(patchwork)
library(emmeans)

# --------------------------------------------------
# Use the 20-min restricted dataframe for final analysis
# --------------------------------------------------

hab_epoch_df <- hab_epoch_df_20min

# --------------------------------------------------
# Output folder
# --------------------------------------------------

out_dir <- fig_dir_str
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

# --------------------------------------------------
# Prepare factors
# --------------------------------------------------

hab_epoch_df$animal <- factor(hab_epoch_df$animal)
hab_epoch_df$animal_day <- factor(hab_epoch_df$animal_day)

# --------------------------------------------------
# Primary outcomes
# --------------------------------------------------

primary_outcomes <- c(
  "frac_moving",
  "frac_forward",
  "mean_speed_path_cms"
)

primary_outcomes <- primary_outcomes[primary_outcomes %in% names(hab_epoch_df)]

outcome_labels <- c(
  frac_moving = "Fraction moving",
  frac_forward = "Fraction forward",
  mean_speed_path_cms = "Speed (cm/s)"
)

# --------------------------------------------------
# Helper functions
# --------------------------------------------------

format_p <- function(p) {
  if (is.na(p)) {
    return("p = NA")
  } else if (p < 0.001) {
    return("p < 0.001")
  } else {
    return(paste0("p = ", sprintf("%.3f", p)))
  }
}

predict_fixed_ci <- function(model, newdata) {
  fixed_formula <- formula(model, fixed.only = TRUE)
  X <- model.matrix(delete.response(terms(fixed_formula)), newdata)
  beta <- fixef(model)
  V <- vcov(model)

  fit <- as.numeric(X %*% beta)
  se <- sqrt(diag(X %*% V %*% t(X)))

  newdata$fit <- fit
  newdata$se <- se
  newdata$lower <- fit - 1.96 * se
  newdata$upper <- fit + 1.96 * se

  return(newdata)
}

theme_fig <- function() {
  theme_classic(base_size = 7) +
    theme(
      axis.title = element_text(size = 7),
      axis.text = element_text(size = 6),
      legend.title = element_text(size = 7),
      legend.text = element_text(size = 6),
      legend.position = "bottom",
      legend.key.width = unit(0.35, "in"),
      plot.title = element_blank(),
      plot.subtitle = element_blank(),
      plot.margin = margin(3, 4, 3, 3)
    )
}

# --------------------------------------------------
# Exposure levels for visualization
# --------------------------------------------------

exposure_min <- min(hab_epoch_df$exposure_session, na.rm = TRUE)
exposure_mean <- mean(hab_epoch_df$exposure_session, na.rm = TRUE)
exposure_max <- max(hab_epoch_df$exposure_session, na.rm = TRUE)

exposure_levels <- data.frame(
  exposure_session = c(exposure_min, exposure_mean, exposure_max),
  exposure_label = c("Early exposure", "Mean exposure", "Late exposure")
)

session_time_seq <- seq(
  min(hab_epoch_df$session_time_min, na.rm = TRUE),
  max(hab_epoch_df$session_time_min, na.rm = TRUE),
  length.out = 100
)

session_time_mean <- mean(hab_epoch_df$session_time_min, na.rm = TRUE)

# --------------------------------------------------
# Fit models and plot
# --------------------------------------------------

model_list <- list()
stats_list <- list()
plot_list <- list()

for (outcome in primary_outcomes) {

  label_i <- unname(outcome_labels[outcome])

  cat("\n\n==============================\n")
  cat("Fitting:", outcome, "\n")
  cat("==============================\n")

  formula_text <- paste0(
    outcome,
    " ~ exposure_session_c * session_time_10m_c + ",
    "(1 | animal) + (1 | animal_day)"
  )

  m <- lmer(
    as.formula(formula_text),
    data = hab_epoch_df,
    REML = FALSE,
    control = lmerControl(
      optimizer = "bobyqa",
      optCtrl = list(maxfun = 2e5)
    )
  )

  model_list[[outcome]] <- m

  coef_table <- as.data.frame(summary(m)$coefficients)

  tmp_stats <- data.frame(
    outcome = outcome,
    y_label = label_i,
    term = rownames(coef_table),
    beta = coef_table[, "Estimate"],
    se = coef_table[, "Std. Error"],
    df = coef_table[, "df"],
    t = coef_table[, "t value"],
    p = coef_table[, "Pr(>|t|)"],
    row.names = NULL
  )

  stats_list[[outcome]] <- tmp_stats

  pred_df <- expand.grid(
    session_time_min = session_time_seq,
    exposure_session = exposure_levels$exposure_session
  ) %>%
    left_join(exposure_levels, by = "exposure_session") %>%
    mutate(
      exposure_session_c = exposure_session - exposure_mean,
      session_time_10m_c = (session_time_min - session_time_mean) / 10
    )

  pred_df <- predict_fixed_ci(m, pred_df)

  pred_df$exposure_label <- factor(
    pred_df$exposure_label,
    levels = c("Early exposure", "Mean exposure", "Late exposure")
  )

  # Raw binned means for visual reference
  q1 <- quantile(hab_epoch_df$exposure_session, 1/3, na.rm = TRUE)
  q2 <- quantile(hab_epoch_df$exposure_session, 2/3, na.rm = TRUE)

  raw_bin_df <- hab_epoch_df %>%
    mutate(
      session_time_bin = floor(session_time_min),
      exposure_label = case_when(
        exposure_session <= q1 ~ "Early exposure",
        exposure_session >= q2 ~ "Late exposure",
        TRUE ~ "Mean exposure"
      )
    ) %>%
    group_by(exposure_label, session_time_bin) %>%
    summarise(
      y_mean = mean(.data[[outcome]], na.rm = TRUE),
      .groups = "drop"
    )

  raw_bin_df$exposure_label <- factor(
    raw_bin_df$exposure_label,
    levels = c("Early exposure", "Mean exposure", "Late exposure")
  )

  # Clean final plot: raw binned means + model-predicted lines.
  # Interaction statistics are reported in the Results/caption, not inside the panel.
  p <- ggplot() +

    geom_point(
      data = raw_bin_df,
      aes(
        x = session_time_bin,
        y = y_mean,
        shape = exposure_label
      ),
      alpha = 0.45,
      size = 0.75
    ) +

    geom_line(
      data = pred_df,
      aes(
        x = session_time_min,
        y = fit,
        linetype = exposure_label
      ),
      linewidth = 0.7
    ) +

    labs(
      x = "Within-session time (min)",
      y = label_i,
      linetype = "Exposure",
      shape = "Exposure"
    ) +

    theme_fig()

  plot_list[[outcome]] <- p
}

hab_epoch_stats_df <- bind_rows(stats_list)

# --------------------------------------------------
# Simple slopes for speed using Satterthwaite df
# --------------------------------------------------

emm_options(
  lmer.df = "satterthwaite",
  lmerTest.limit = nrow(hab_epoch_df) + 1000,
  pbkrtest.limit = nrow(hab_epoch_df) + 1000
)

speed_model <- model_list[["mean_speed_path_cms"]]

exposure_c_values <- c(
  exposure_min - exposure_mean,
  0,
  exposure_max - exposure_mean
)

speed_trends <- emtrends(
  speed_model,
  ~ exposure_session_c,
  var = "session_time_10m_c",
  at = list(exposure_session_c = exposure_c_values),
  lmer.df = "satterthwaite"
)

hab_speed_simple_slopes_t_df <- as.data.frame(
  summary(
    speed_trends,
    infer = c(TRUE, TRUE),
    null = 0
  )
)

hab_speed_simple_slopes_t_df$exposure_label <- c(
  "Early exposure",
  "Mean exposure",
  "Late exposure"
)

print(hab_epoch_stats_df)
print(hab_speed_simple_slopes_t_df)

# --------------------------------------------------
# Combine and save figure
# --------------------------------------------------

p_combined <- plot_list[[primary_outcomes[1]]] +
  plot_list[[primary_outcomes[2]]] +
  plot_list[[primary_outcomes[3]]] +
  plot_layout(nrow = 1, guides = "collect") &
  theme(legend.position = "bottom")

print(p_combined)

ggsave(
  filename = file.path(out_dir, "Fig1_within_session_habituation_3panel.pdf"),
  plot = p_combined,
  width = 7,
  height = 2.1
)

ggsave(
  filename = file.path(out_dir, "Fig1_within_session_habituation_3panel.png"),
  plot = p_combined,
  width = 7,
  height = 2.1,
  dpi = 600
)

# --------------------------------------------------
# Save tables
# --------------------------------------------------

write.csv(
  hab_epoch_stats_df,
  file.path(out_dir, "habituation_epoch_session_time_LME_stats.csv"),
  row.names = FALSE
)

write.csv(
  hab_speed_simple_slopes_t_df,
  file.path(out_dir, "habituation_speed_simple_slopes_early_mean_late_exposure.csv"),
  row.names = FALSE
)

## 15. Inspect final simple slopes for speed

The speed interaction is followed up using simple slopes of within-session time at early, mean, and late exposure. These slopes use Satterthwaite degrees of freedom so the manuscript can report `t(df)` values.

In [ ]:
display(hab_speed_simple_slopes_t_df)

print(hab_speed_simple_slopes_t_df.columns.tolist())